# Session 1: When Optimal Fails, Stress-Testing Classical Minimum-Variance Portfolios

Modern portfolio theory promises an "optimal" allocation, but optimal under what assumptions? In this session, we'll build a classical minimum-variance portfolio and then systematically break it. We'll use AI-driven hybrid Monte Carlo simulation to expose how fragile textbook solutions can be when the world doesn't cooperate.

> __Frictionless Assumptions:__
>
> In session 1, we assume zero transaction costs, infinitely divisible shares, perfect order fills at each day's stated price, no taxes, no slippage, no margin or borrowing limits beyond the long-only constraint, and instant execution. 

These assumptions are pedagogical scaffolding, like the frictionless plane in physics. They let us see the structural ideas (Markowitz optimization, hybrid forward simulation, distributional scorecards) without the operational noise of real-world trading. Later, we introduce transaction frictions, and we see where they matter.

> __Learning Objectives:__
>
> * __From Prices to Portfolios:__ Derive log growth rates from price data, then estimate portfolio reward and risk using the expected growth rate vector and covariance matrix. Use the Single Index Model (SIM) to construct a factor-based covariance matrix with bootstrap uncertainty quantification.
> * __Optimization and the Sharpe Ratio:__ Solve the Markowitz minimum-variance quadratic program and trace the efficient frontier. Find the tangency (maximum Sharpe ratio) portfolio via second-order cone programming.
> * __Forward Stress Testing and Tail-Risk Scorecard:__ Generate a Monte Carlo ensemble of synthetic futures via the hybrid SIM construction and run buy-and-hold across each path. Score the resulting wealth distribution with formal tail-risk metrics (VaR, CVaR, max drawdown) and the portfolio Net Present Value relative to a risk-free baseline.

Let's get started!
___

## Examples
The following example notebooks accompany this lecture. They contain executable code that implements the concepts discussed here:

>
> * [▶ Let's explore the stylized facts of log growth rate data](eCornell-AI-Finance-S1-Example-StylizedFacts-Example-May-2026.ipynb). We compute growth rates from the synthetic training dataset, test for fat tails via Anderson-Darling and Hill's estimator, and verify volatility clustering via autocorrelation of absolute returns.
> * [▶ Let's build a classical minimum-variance portfolio from synthetic data](eCornell-AI-Finance-S1-Example-BuildMinVariancePortfolio-May-2026.ipynb). We generate a synthetic asset universe, estimate $\mathbb{E}[\mathbf{g}]$ and $\boldsymbol{\Sigma}_g$, solve the quadratic program, compute the efficient frontier, and explore input sensitivity.
> * [▶ Let's stress-test the minimum-variance portfolio](eCornell-AI-Finance-S1-Example-StressTestMinVariancePortfolio-May-2026.ipynb). We generate 5,000 synthetic futures via the hybrid SIM construction, run buy-and-hold across four portfolios (min-var, equal-weight 1/N, the market index, and a continuously compounded risk-free baseline), and produce a tail-risk scorecard featuring VaR, CVaR, max drawdown, and portfolio NPV against the risk-free baseline.
> * [▶ Let's estimate SIM parameters with bootstrap uncertainty quantification](eCornell-AI-Finance-S1-Example-SIMParameterEstimation-May-2026.ipynb). We estimate Single Index Model parameters from the synthetic training dataset via regularized OLS, bootstrap the sampling distribution for a demonstration ticker, and then run batch estimation across all 424 tickers and save the results.

___

## From Prices to Growth Rates
The inputs to portfolio optimization, expected growth rates and covariances, must be _estimated_ from observed price data, starting with the __Continuously Compounded Growth Rate (CCGR)__, which converts a time series of asset prices into a stationary series of growth rates suitable for statistical analysis:

>  __Continuously Compounded Growth Rate (CCGR)__
>
> Let's assume a model of the share price of firm $i$ is governed by an expression of the form:
>$$
\begin{align*}
S^{(i)}_{j} &= S^{(i)}_{j-1}\;\exp\left(\underbrace{g^{(i)}_{j,j-1}\Delta{t}_{j}}_{\text{return}}\right)
\end{align*}
> $$
> where $S^{(i)}_{j-1}$ denotes the share price of firm $i$ at time index $j-1$, $S^{(i)}_{j}$ denotes the share price of firm $i$ at time index $j$, and $\Delta{t}_{j} = t_{j} - t_{j-1}$ denotes the length of a time step (units: years) between time index $j-1$ and $j$. 

The value we are going to estimate from data is the growth rate $g^{(i)}_{j,j-1}$ (units: inverse years) for each firm $i$. Let's rearrange the continuous share price expression to solve for $g^{(i)}_{j,j-1}$:
$$\begin{align*}
S^{(i)}_{j} & = S^{(i)}_{j-1}\;\exp\left(g^{(i)}_{j,j-1}\Delta{t}_{j}\right) \\
S^{(i)}_{j}/S^{(i)}_{j-1} & = \exp\left(g^{(i)}_{j,j-1}\Delta{t}_{j}\right) \\
\ln\left(S^{(i)}_{j}/S^{(i)}_{j-1}\right) & = g^{(i)}_{j,j-1}\Delta{t}_{j} \\
g^{(i)}_{j,j-1} & = \left(\frac{1}{\Delta{t}_{j}}\right)\,\ln\left(\frac{S^{(i)}_{j}}{S^{(i)}_{j-1}}\right)\quad\blacksquare
\end{align*}
$$

Log growth rates are additive over time and approximate simple percentage returns for small values. Given $T$ observations for each of $N$ assets, we assemble the growth rates into a _data matrix_ $\mathbf{X} \in \mathbb{R}^{(T-1) \times N}$ where each row is a time step and each column is an asset. This matrix is the raw material for estimating the inputs to the optimizer.

> __Example__
> 
> Growth rate distributions have some interesting statistical properties (stylized facts) worth exploring:
>
> * __Heavy (fat) tailed distribution__: Extreme price movements are more likely than a Normal distribution predicts.
> * __Absence of Autocorrelation__: Returns are approximately uncorrelated, consistent with a random walk with occasional jumps.
> * __Volatility clustering__: Large price movements tend to be followed by other large moves, and small moves follow small moves.
>
> [▶ Let's explore the stylized facts of log growth rate data](eCornell-AI-Finance-S1-Example-StylizedFacts-Example-May-2026.ipynb). We compute growth rates from the synthetic training dataset, test for fat tails via Anderson-Darling and Hill's estimator, and verify volatility clustering via autocorrelation of absolute returns. For a deeper dive into the stylized facts of growth rate data: [check out the stylized facts deeper dive notebook](eCornell-AI-Finance-S1-DeeperDive-StylizedFacts-May-2026.ipynb). 

___

<div>
    <center>
        <img src="figs/Fig-MinVar-Portfolio-RA-RiskFree-Schematic.svg" width="980"/>
    </center>
</div>

## Modern Portfolio Theory (MPT)
In 1952, [Harry Markowitz](https://doi.org/10.2307/2975974) proposed a deceptively simple idea: investors care about the _expected return_ and the _variance_ (risk) of their portfolio, and they want the best trade-off between the two. The minimum-variance portfolio is the allocation that achieves the lowest possible risk for a given level of expected return.

> __Reference:__ Modern Portfolio Theory was introduced by Harry Markowitz in the 1950s and has since become a foundational concept in finance. Markowitz was awarded the Nobel Prize in Economic Sciences in 1990 for his pioneering work in this area. The original publication is given here: [Portfolio Selection, The Journal of Finance, Vol. 7, No. 1 (Mar., 1952), pp. 77-91](https://www.jstor.org/stable/2975974). The Nobel Prize information is available here: [The Sveriges Riksbank Prize in Economic Sciences in Memory of Alfred Nobel 1990](https://www.nobelprize.org/prizes/economic-sciences/1990/markowitz/facts/).
>
> The key idea of MPT is to __optimally balance risk and reward__ by diversifying investments across different assets. Moving along the frontier from Portfolio 2 to Portfolio 1 increases both reward _and_ risk. The curvature of the frontier encodes the _cost of chasing return_: initially, a small increase in risk buys a lot of extra return, but the marginal benefit diminishes as we move further right.

### Portfolio Reward
The reward of a portfolio is measured by its expected growth (return), which is the weighted sum of the expected growth rates (returns) of the individual assets in the portfolio weighted by the fraction of the total portfolio value invested in each asset. 

Suppose we have a portfolio $\mathcal{P}$ consisting of $N$ assets. Let $w_i\in\mathbb{R}_{\geq 0}$ be the weight of asset $i$ in the portfolio (i.e., the dollar fraction of the total portfolio value invested in asset $i$), and let $\mathbb{E}[g_i]$ be the expected growth rate (scaled return) of asset $i$. Then, the expected growth rate (return) of the portfolio, denoted as $\mathbb{E}[g_{\mathcal{P}}]$, is given by:
$$
\mathbb{E}[g_{\mathcal{P}}] = \sum_{i=1}^{N} w_i \mathbb{E}[g_i]\quad\Longrightarrow\;\mathbf{w}^{\top}\mathbb{E}[\mathbf{g}]
$$
where $N$ is the total number of assets in the portfolio, i.e., $|\mathcal{P}| = N$, the weight vector is $\mathbf{w}^{\top} = [w_1, w_2, \dots, w_N]$, the sum of weights is one, and $\mathbb{E}[\mathbf{g}] = [\mathbb{E}[g_1], \mathbb{E}[g_2], \dots, \mathbb{E}[g_N]]^{\top}$ is the vector of expected growth rates (returns) of the individual assets.

> __Growth Rate versus Return?__  
>
> We use a somewhat strange convention of measuring reward in terms of the expected growth rate $\mathbb{E}[g_i]$ of asset $i$ instead of the more typical expected return $\mathbb{E}[r_i]$. However, regardless of this choice, the reward argument remains the same (it's just scaled by the inverse time step between the two perspectives). 
> 
> Let $g_{\star} = r_{\star}/\Delta{t}$. Then, the expected return of the portfolio is given by:
> $$
\begin{align*}
\mathbb{E}[g_{\mathcal{P}}] & = \sum_{i=1}^{N} w_i\,\mathbb{E}[g_i]\\
& = \sum_{i=1}^{N} w_i\,\mathbb{E}\Bigl[\frac{r_i}{\Delta{t}}\Bigr]\quad\Longrightarrow\text{pull out } \left(\frac{1}{\Delta{t}}\right)\text{ from the sum} \\
& = \left(\frac{1}{\Delta{t}}\right)\,\sum_{i=1}^{N} w_i\,\mathbb{E}[r_i]\\
& = \left(\frac{1}{\Delta{t}}\right)\mathbf{w}^{\top}\mathbb{E}[\mathbf{r}]\quad\blacksquare    
\end{align*}$$
> The expected return $\mathbf{w}^{\top}\mathbb{E}[\mathbf{r}]$ is __dimensionless__, while the expected growth rate of the portfolio has units of $[\text{time}]^{-1}$. We can convert between the two by multiplying or dividing by the time step $\Delta{t}$.

### Portfolio Risk
The risk of a portfolio is (typically) measured by its variance (or standard deviation) of growth rates (returns), which takes into account the variances of the individual assets as well as the covariances between them. The portfolio variance written in terms of growth rates is given by:
$$
\text{Var}(g_{\mathcal{P}}) = \sum_{i=1}^{N} \sum_{j=1}^{N} w_i w_j \text{Cov}(g_i, g_j)\quad\Longrightarrow\;\mathbf{w}^{\top}\boldsymbol{\Sigma}_{g}\mathbf{w}
$$
where $\text{Cov}(g_i, g_j)$ is the covariance between the growth rates of assets $i$ and $j$, and $\boldsymbol{\Sigma}_{g}$ is the covariance matrix of asset growth rates, defined as:
$$
\boldsymbol{\Sigma}_{g} =
\begin{bmatrix}
\text{Var}(g_1) & \text{Cov}(g_1, g_2) & \cdots & \text{Cov}(g_1, g_N) \\
\text{Cov}(g_2, g_1) & \text{Var}(g_2) & \cdots & \text{Cov}(g_2, g_N) \\
\vdots & \vdots & \ddots & \vdots \\
\text{Cov}(g_N, g_1) & \text{Cov}(g_N, g_2) & \cdots & \text{Var}(g_N)
\end{bmatrix}
$$

However, we could also write the portfolio risk with respect to the returns instead of the growth rates, i.e., $\text{Cov}(r_i, r_j)$ and $\text{Var}(r_{\mathcal{P}})$.

> __Covariance of Growth Rates versus Returns__  
> 
> The portfolio risk argument remains the same. Let $g_{\star} = r_{\star}/\Delta{t}$. Then, the variance of the portfolio of $N$ assets is given by:
> $$
\begin{align*}
\text{Var}(g_{\mathcal{P}}) & = \sum_{i=1}^{N} \sum_{j=1}^{N} w_i w_j \text{Cov}(g_i, g_j)\\
& = \sum_{i=1}^{N} \sum_{j=1}^{N} w_i w_j \text{Cov}\Bigl(\frac{r_i}{\Delta{t}}, \frac{r_j}{\Delta{t}}\Bigr)\quad\Longrightarrow\text{pull out } \frac{1}{(\Delta{t})^2} \\
& = \frac{1}{(\Delta{t})^2} \sum_{i=1}^{N} \sum_{j=1}^{N} w_i w_j \text{Cov}(r_i, r_j)\\
& = \left(\frac{1}{(\Delta{t})^2}\right)\mathbf{w}^{\top}\boldsymbol{\Sigma}_{r}\mathbf{w}\quad\blacksquare    
\end{align*}$$
> where $\boldsymbol{\Sigma}_{r}$ is the covariance matrix of asset returns. The portfolio variance $\mathbf{w}^{\top}\boldsymbol{\Sigma}_{r}\mathbf{w}$ is __dimensionless__, while the portfolio variance of growth rates has units of $[\text{time}]^{-2}$. We can convert between the two by multiplying or dividing by $(\Delta{t})^2$. 

However, there is a catch. We use (by convention) the _annualized_ variance of the portfolio, which means we need to multiply the portfolio variance by the number of time steps in a year, i.e., $N_{\text{steps}} = 1/\Delta{t}$. Thus, the annualized portfolio variance is given by:
$$
\boxed{
\begin{align*}
\text{Var}_{\text{annualized}}(g_{\mathcal{P}}) = \left(\frac{1}{\Delta{t}}\right)\mathbf{w}^{\top}\boldsymbol{\Sigma}_{r}\mathbf{w}\quad\blacksquare
\end{align*}
}
$$

Ok, we have our risk and reward measures for a portfolio of $N$ assets. Now, let's see how we can use this information to construct an optimal portfolio that balances these two forces.


### Optimal Weights for a Risky Portfolio
The goal of MPT is to find the optimal weights $\mathbf{w}$ that minimize the portfolio risk for a given level of expected return, or equivalently, maximize the expected return for a given level of risk. This is typically done by solving a __constrained quadratic programming__ problem that has a unique global minimum and can be solved efficiently using solvers such as [Ipopt](https://coin-or.github.io/Ipopt/) or [COSMO](https://github.com/oxfordcontrol/COSMO.jl). 

Let's consider the case when we have a portfolio $\mathcal{P}$ consisting of $N$ __risky assets__, i.e., only equity, ETFs (or potentially derivatives) but no fixed income assets. In this case, we can formulate the optimal weights optimization problem (written in terms of growth rate) as:
$$
\boxed{
\begin{align*}
\text{minimize}~\text{Var}(g_{\mathcal{P}}) &= \sum_{i\in\mathcal{P}}\sum_{j\in\mathcal{P}}w_{i}w_{j}\underbrace{\text{Cov}\left(g_{i},g_{j}\right)}_{= \sigma_{i}\sigma_{j}\rho_{ij}}\quad{\Longleftrightarrow\mathbf{w}^\top \boldsymbol{\Sigma}_{g} \mathbf{w}} \\
\text{subject to}~\mathbb{E}[g_{\mathcal{P}}]& =  \sum_{i\in\mathcal{P}}w_{i}\;\mathbb{E}[g_{i}] = R^{*}\quad\Longleftrightarrow\mathbf{w}^\top \mathbb{E}[\mathbf{g}] = R^{*} \\
\sum_{i\in\mathcal{P}}w_{i} & =  1 \\
l_{i} & \leq w_{i} \leq u_{i} \quad \forall\, i\in\mathcal{P}
\end{align*}}
$$
The objective $\mathbf{w}^{\top}\boldsymbol{\Sigma}_{g}\,\mathbf{w}$ is the portfolio variance (risk). The first constraint sets the target return at $R^{*}$. The budget constraint forces the weights to sum to 1 (fully invested). The box constraints enforce position limits.

__Problem setup__: The term $R^{*}$ is the investor’s target annualized return for portfolio $\mathcal{P}$, while $l_i$ and $u_i$ are the lower and upper bounds on each weight; for example, no short selling implies $l_i = 0$. The weights must sum to one so the portfolio is fully invested, though these constraints can be relaxed if short selling is allowed. 

To construct the efficient frontier, we vary $R^{*}$ and solve the optimization problem for each target return, producing the set of minimum variance portfolios that form the frontier, with each point corresponding to a different set of portfolio weights. 

Let’s work through an example.

> __Example__
> 
> [▶ Let's use artificial intelligence (AI) to select to select risky tickers for our portfolio](eCornell-AI-Finance-S1-Example-BuildMinVariancePortfolio-RA-May-2026.ipynb). Let's build a classical minimum-variance portfolio from synthetic data. We generate a synthetic asset universe, estimate the growth rates and the covariance matrix, solve the quadratic program, compute the efficient frontier, and explore input sensitivity. We'll let AI select the tickers. 
___

### The Capital Allocation Line and Tangent Portfolio

When we introduce a risk-free asset into our portfolio optimization problem, something remarkable happens: the efficient frontier transforms into a straight line in risk-return space. This line is called the __Capital Allocation Line (CAL)__.

The Capital Allocation Line describes all portfolios that can be formed by combining a risk-free asset (e.g., Treasury STRIPS) with return $g_f$ and zero variance, and a single optimal risky portfolio called the __tangent portfolio__ (denoted by $T$). The tangent portfolio has expected return $\mathbb{E}[g_T]$ and variance $\sigma_T^2$. Any portfolio on the CAL can be expressed as:
$$
\begin{align*}
\mathbb{E}[g_{\mathcal{P}}] &= w_f g_f + (1 - w_f)\;\mathbb{E}[g_T]\\
\sigma_{\mathcal{P}} &= (1 - w_f) \sigma_T
\end{align*}
$$
where $w_f$ is the fraction invested in the risk-free asset, $(1-w_f)$ is the fraction invested in the tangent portfolio, $\sigma_{\mathcal{P}} = \sqrt{\operatorname{Var}(g_{\mathcal{P}})}$ is the standard deviation of the portfolio return, and $\sigma_T = \sqrt{\operatorname{Var}(g_T)}$ is the standard deviation of the tangent portfolio return. To derive the CAL equation, we solve for $w_f$ from the second equation and substitute into the first:
$$
\begin{align*}
\sigma_{\mathcal{P}} &= (1 - w_f) \sigma_T\\
\frac{\sigma_{\mathcal{P}}}{\sigma_T} &= 1 - w_f\\
w_f &= 1 - \frac{\sigma_{\mathcal{P}}}{\sigma_T}
\end{align*}
$$
Substituting this expression for $w_f$ into the expected return equation:
$$
\begin{align*}
\mathbb{E}[g_{\mathcal{P}}] &= w_f g_f + (1 - w_f) \mathbb{E}[g_T]\\
&= \left(1 - \frac{\sigma_{\mathcal{P}}}{\sigma_T}\right) g_f + \frac{\sigma_{\mathcal{P}}}{\sigma_T} \mathbb{E}[g_T]\\
&= g_f - \frac{\sigma_{\mathcal{P}}}{\sigma_T} g_f + \frac{\sigma_{\mathcal{P}}}{\sigma_T} \mathbb{E}[g_T]\\
&= g_f + \frac{\sigma_{\mathcal{P}}}{\sigma_T} \left(\mathbb{E}[g_T] - g_f\right)\\
&= g_f + \underbrace{\left(\frac{\mathbb{E}[g_T] - g_f}{\sigma_T}\right)}_{\text{Sharpe ratio T.P.}}\;\sigma_{\mathcal{P}}\quad\blacksquare
\end{align*}
$$
This is a __linear relationship__ between expected return and risk. The slope of this line is the __Sharpe ratio of the tangent portfolio__, which measures the excess return per unit of risk for that optimal risky portfolio.

To find the tangent portfolio, we identify the risky portfolio that maximizes the Sharpe ratio. Geometrically, it's the point where a line from the risk-free rate is tangent to the risky-only efficient frontier. Using the single index model, where $g_{\mathrm{mkt}}$ denotes the market growth rate, we solve the optimization problem for the weights $w_i$ of the risky assets in portfolio $\mathcal{P}$ that maximizes the Sharpe ratio:
$$
\boxed{
\begin{align*}
\text{maximize} &\quad \frac{\mathbb{E}[g_{\mathcal{P}}] - g_f}{\sigma_{\mathcal{P}}} = \frac{\alpha_{\mathcal{P}} + \beta_{\mathcal{P}}\;\mathbb{E}[g_{\mathrm{mkt}}] - g_f}{\sigma_{\mathcal{P}}}\\
\text{subject to} &\quad \sum_{i\in\mathcal{P}}w_{i} = 1\\
&\quad w_{i} \geq 0 \qquad \forall{i}\in\mathcal{P}
\end{align*}}
$$
The portfolio $\mathcal{P}$ that solves this optimization problem is the tangent portfolio $T$. Once found, its Sharpe ratio $\left(\mathbb{E}[g_T] - g_f\right)/{\sigma_T}$ becomes the slope of the CAL.

> __Practical Notes__
>
> * __Concentration constraints:__ In practice, unconstrained tangent portfolio optimization often produces extreme weights in a few assets, especially with imprecise estimates. Practitioners commonly impose concentration constraints like $0 \leq w_i \leq u$ to force diversification. While these constraints theoretically reduce the Sharpe ratio, they often improve out-of-sample performance by making portfolios more robust to estimation error.
> * __Tricky problem:__ Maximizing the Sharpe ratio turns out to be surprisingly tricky, because the objective function is a ratio of two functions that depend on the weights $w_i$ (with a quadratic in the denominator). The approach that we use to solve this problem is to transform it in a Second Order Cone Program (SOCP), which can be solved efficiently using modern optimization software, such [as the COSMO.jl package in Julia](https://github.com/oxfordcontrol/COSMO.jl.git).
> * __The Sharpe ratio is more general than just the Tangent Portfolio__. Any portfolio risky-asset $\mathcal{P}$ has its own Sharpe ratio, which is a measure of the risk-return trade-off of that specific portfolio. Thus, the Sharpe ratio provides a standardized (dimensionless) way to compare different portfolios or investment strategies, regardless of their individual compositions.

Let's look at an example that demonstrates constructing the Capital Allocation Line and finding the tangent portfolio.

> __Example__
> 
> [▶ Let's use artificial intelligence (AI) to select to compute the capital allocation line](eCornell-AI-Finance-S1-Example-MinVariancePortfolio-RRFA-May-2026.ipynb). Let's use AI to select some tickers, and then compute the capital allocation line and find the tangent portfolio. We'll use the single index model to estimate the covariance matrix, and then solve the second-order cone program to find the optimal weights for the tangent portfolio.
>  
> For more information on the single index model (SIM) and how to estimate its parameters, check out the [SIM parameter estimation example notebook](eCornell-AI-Finance-S1-Example-SIMParameterEstimation-May-2026.ipynb).

___

## When Optimal Fails: Weaknesses of Mean-Variance Optimization
The minimum-variance framework is elegant, but in practice it has three persistent weaknesses:

* **Input sensitivity:** The optimizer treats $\mathbb{E}[\mathbf{g}]$ and $\boldsymbol{\Sigma}_g$ as if they were known exactly, even though both are estimated from historical data. Small estimation errors can produce large changes in optimal weights, which is why [Michaud described mean-variance optimization as an _"error maximizer"_](https://doi.org/10.2469/faj.v45.n1.31): it tends to overweight assets with overstated returns and underweight those with understated returns. Errors in expected returns are especially damaging; [Chopra and Ziemba found they are far more consequential than errors in variances](https://doi.org/10.3905/jpm.1993.409440)

* **Weight concentration:** Without meaningful constraints, minimum-variance portfolios often concentrate in a small number of assets. The quadratic objective $\mathbf{w}^{\top}\boldsymbol{\Sigma}_g\mathbf{w}$ strongly rewards slightly lower variance, so a portfolio that appears diversified can end up placing most of its weight in just a few names.

* **Assumption fragility:** The framework assumes returns are drawn from a stable multivariate distribution, but that assumption often fails when it matters most. During crises and regime shifts, correlations rise, volatilities jump, and expected returns change, so a portfolio that looked optimal in calm markets can perform poorly when conditions shift.

### Forward Stress Testing via Hybrid Monte Carlo

The classical approach to stress testing perturbs the inputs (correlations, prices, costs) and re-solves the optimization. We take a different approach: we fix the allocation and instead sample many *forward futures* from a calibrated generative model. The optimization itself is no longer the variable; the world is. Given an allocation $\mathbf{w}$ and initial wealth $W_{\mathcal{P}}(0) = B_0$, the share count purchased in asset $i$ at time $t_0$ is fixed by the allocation:
$$\boxed{n_i = \frac{w_i\,W_{\mathcal{P}}(0)}{S_i(t_0)}, \qquad i = 1,\ldots,N}$$
These shares are held unchanged for the entire horizon (no rebalancing — Session 1 is _frictionless and fixed-weight_). The portfolio wealth at any future time $t_k$ is then the share-weighted price aggregate:
$$\boxed{W_{\mathcal{P}}(t_k) = \sum_{i=1}^{N} n_i\,S_i(t_k)}$$

We draw $n_{\text{paths}}$ synthetic price trajectories $\bigl\{S_i^{(p)}(t_k)\bigr\}_{p=1}^{n_{\text{paths}}}$ from the calibrated **hybrid single index hidden Markov model (HMM)**: Each synthetic path produces a terminal wealth given by:

$$W_{\mathcal{P}}^{(p)}(T) = \sum_{i=1}^{N} n_i\,S_i^{(p)}(T)$$

and the collection $\bigl\{W_{\mathcal{P}}^{(p)}(T)\bigr\}_{p=1}^{n_{\text{paths}}}$ is the **empirical distribution of outcomes** that the rest of the scorecard works with.

> __Why a distribution, not a number?__
>
> A point estimate of expected return tells us the _center_ of the wealth distribution and nothing about its shape. Tail risk lives in the _tails_, i.e., the worst 5% of outcomes, and you cannot see it without sampling many paths. The hybrid Monte Carlo turns "what is the expected outcome?" into "what does the full distribution of outcomes look like, and how bad is the bad tail?"
___


## Portfolio NPV and Tail-Risk Metrics

A purely nominal comparison: _"did the portfolio make money?"_, sets the bar too low. The economically meaningful question is: _did the portfolio beat what the same dollars would have earned sitting in a risk-free or alternative asset?_ The right tool for that comparison is the **Net Present Value (NPV)** of the portfolio.

Let's define the NPV of a portfolio with constant continuous-compounding discount rate $\bar r$ over horizon $T$ as the expression:

$$\boxed{\text{NPV}(\bar r, T) = -W_{\mathcal{P}}(0) + W_{\mathcal{P}}(T)\,e^{-\bar r\,T}}$$

The first term is the cash outflow at $t_0$ (the initial investment); the second is the terminal portfolio wealth discounted back to today's dollars. Dividing by the initial wealth gives the **scaled NPV**, the fractional excess in present value per dollar invested:

$$\boxed{\frac{\text{NPV}(\bar r, T)}{W_{\mathcal{P}}(0)} = \frac{W_{\mathcal{P}}(T)}{W_{\mathcal{P}}(0)}\,e^{-\bar r\,T} - 1}$$

For stress-test evaluation we set $\bar r = r_f$, the **risk-free rate**. With this choice the NPV has a sharp economic interpretation:

> __Interpretation of NPV with risk-free discounting:__
>
> - $\text{NPV} > 0$: the risky portfolio **beat** a risk-free zero-coupon investment in present-value terms.
> - $\text{NPV} < 0$: the risky portfolio **underperformed** the risk-free baseline; you would have done better holding cash earning $r_f$.
> - $\text{NPV} = 0$: by construction this is the risk-free portfolio itself, $W_{\mathcal{P}}(T) = W_{\mathcal{P}}(0)\,e^{r_f\,T}$. It is the *zero we are measuring against*.

Because the hybrid Monte Carlo gives us a distribution $\{W_{\mathcal{P}}^{(p)}(T)\}$, NPV is also a distribution $\{\text{NPV}^{(p)}\}$. One important metric is the NPV failure rate, the fraction of paths on which the portfolio underperformed the risk-free baseline:

> __NPV-fail rate:__
>
> The economically meaningful failure probability is the fraction of paths on which the portfolio underperformed the risk-free baseline:
>
> $$\boxed{P\bigl[\text{NPV} < 0\bigr] = P\Bigl[W_{\mathcal{P}}(T) < W_{\mathcal{P}}(0)\,e^{r_f\,T}\Bigr]}$$
>
> This strictly dominates the nominal-loss rate $P[W_T < W_{\mathcal{P}}(0)]$ as a measure of failure. A portfolio that grew by 1% nominal over a 5% risk-free year has _failed_ even though it didn't lose money in nominal terms — its capital was idle while the risk-free alternative was earning 4 points more.



### Tail-Risk Metrics

Alongside NPV, the terminal-wealth distribution $\bigl\{W_{\mathcal{P}}^{(p)}(T)\bigr\}$ is summarized by a small set of metrics that focus on the **left tail** — the bad outcomes — rather than the center.

> __Value-at-Risk (VaR) at level $\alpha$:__
>
> The Value-at-Risk at level $\alpha$ is the wealth threshold below which only an $\alpha$ fraction of paths end:
>
> $$\boxed{\text{VaR}_{\alpha}(W_T) = \inf\bigl\{w \in \mathbb{R} : \,P\bigl(W_T \leq w\bigr) \geq \alpha\bigr\}}$$
>
> Equivalently, $\text{VaR}_{\alpha}$ is the $\alpha$-quantile of the terminal-wealth distribution. For $\alpha = 5\%$ it answers: _"what is the worst case I should expect 95% of the time?"_

> __Conditional VaR (CVaR) / Expected Shortfall:__
>
> The Conditional VaR at level $\alpha$ is the **mean** terminal wealth conditional on being in the worst $\alpha$ tail:
>
> $$\boxed{\text{CVaR}_{\alpha}(W_T) = \mathbb{E}\bigl[\,W_T \,\big|\, W_T \leq \text{VaR}_{\alpha}(W_T)\,\bigr]}$$
>
> Where VaR only sees the cutoff, CVaR sees the entire tail. CVaR is also known as **Expected Shortfall (ES)** and is a *coherent* risk measure (it satisfies sub-additivity, which VaR does not), so it is the metric of choice for modern tail-risk reporting.

> __CVaR sample standard error:__
>
> Given $n_{\text{paths}}$ Monte Carlo samples, the tail contains $n_{\text{tail}} = \lfloor \alpha\,n_{\text{paths}}\rfloor$ paths. The plug-in estimator and its analytical standard error are:
>
> $$\boxed{\widehat{\text{CVaR}}_{\alpha} = \frac{1}{n_{\text{tail}}}\sum_{p\,\in\,\text{tail}} W_T^{(p)}, \qquad \text{SE}\bigl(\widehat{\text{CVaR}}_{\alpha}\bigr) \approx \frac{\mathrm{std}(\text{tail})}{\sqrt{n_{\text{tail}}}}}$$
>
> Reporting the standard error alongside the point estimate tells you whether $n_{\text{paths}}$ is large enough to trust the number. A rule of thumb: if the SE-to-CVaR ratio exceeds about 1%, increase $n_{\text{paths}}$ before quoting the figure.

> __Maximum Drawdown:__
>
> Given a path-wise wealth series $W_{\mathcal{P}}^{(p)}(t)$, define the running peak as $\hat{W}^{(p)}(t) = \max_{s \leq t} W_{\mathcal{P}}^{(p)}(s)$. The maximum drawdown along that path is the deepest peak-to-trough fractional decline:
>
> $$\boxed{\text{MaxDD}^{(p)} = \max_{t}\;\frac{\hat{W}^{(p)}(t) - W_{\mathcal{P}}^{(p)}(t)}{\hat{W}^{(p)}(t)}}$$
>
> Drawdown is path-wise, not terminal: a portfolio can end the horizon at break-even after a 30% mid-path drawdown that would have triggered margin calls in real life. The scorecard reports both the median and the 95th percentile of $\text{MaxDD}^{(p)}$ across the $n_{\text{paths}}$ futures.

___

## The Baseline Scorecard

Before we can evaluate any improvements (Sessions 2-4), we need a _baseline_ — a quantitative record of how the classical fixed-weight minimum-variance allocation behaves across the hybrid Monte Carlo ensemble. The scorecard tracks **seven** metrics, computed across all $n_{\text{paths}}$ synthetic futures:

| Metric | Definition | What it tells us |
|:-------|:----------|:---------------|
| __Median NPV__ | $\text{median}_p\,\text{NPV}^{(p)}(r_f, T)$ | Excess wealth over the risk-free baseline (today's dollars) |
| __NPV-fail rate__ | $P\bigl[\text{NPV} < 0\bigr]$ | Fraction of futures the portfolio underperformed risk-free |
| __Median $W_T$__ | $\text{median}_p\,W_{\mathcal{P}}^{(p)}(T)$ | Center of the terminal-wealth distribution |
| __VaR$_5$__ | 5th-percentile terminal wealth | Worst-case threshold for 95% of futures |
| __CVaR$_5$__ (with SE) | Mean wealth in the worst 5% of paths | Average loss in the bad tail, with sampling SE |
| __Max Drawdown__ (median, P95) | Median and 95th-percentile of $\text{MaxDD}^{(p)}$ across paths | Worst-case path-wise loss along the trajectory |
| __Median Sharpe__ | Median of per-path Sharpe ratios | Risk-adjusted return summary |

> __Why these seven and not the textbook four?__
>
> Turnover and trading-cost metrics — staples of classical scorecards — are *zero by construction* in Session 1 because the allocation is fixed for the entire horizon (no rebalancing) and we are operating in the frictionless regime. They become meaningful in [Session 2](../session-2/eCornell-AI-Finance-S2-Lecture-AIRebalancingEngine-May-2026.ipynb) when the AI rebalancing engine introduces time-varying weights and transaction costs. In Session 1's frictionless world, the seven metrics above are necessary and sufficient to characterize a fixed-weight allocation's behavior across many futures.

> __Reading the scorecard:__
>
> The risk-free row of the scorecard is intentionally degenerate: deterministic terminal wealth $B_0\,e^{r_f T}$, zero drawdown, zero Sharpe (zero excess vol), $\text{NPV} = 0$, NPV-fail rate $= 0$. It is not a competitor — it is the *zero line* against which every risky portfolio's NPV is measured. A risky portfolio "wins" relative to risk-free if its median NPV is positive *and* its NPV-fail rate is acceptably low (depending on risk tolerance).

The baseline scorecard is computed in the [stress-test example notebook](eCornell-AI-Finance-S1-Example-StressTestMinVariancePortfolio-May-2026.ipynb) and persisted to disk as the bar that Session 2's adaptive rebalancing engine must clear.

___

## Summary

In this session, we built the full analytical pipeline — from prices to growth rates, through SIM estimation and covariance construction, to minimum-variance and maximum-Sharpe optimization — and then stress-tested the result by running buy-and-hold across thousands of synthetic futures and scoring the wealth distribution against a risk-free baseline.

> __Key Takeaways:__
>
> * __The Single Index Model:__ SIM reduces covariance estimation from quadratic to linear in the number of assets, producing better-conditioned inputs for the optimizer. Bootstrap uncertainty quantification reveals how reliable the alpha and beta estimates actually are.
> * __The tangency portfolio:__ The maximum-Sharpe portfolio sits at the point where the Capital Market Line touches the efficient frontier. It accepts more variance than the min-variance portfolio but delivers substantially better risk-adjusted return.
> * __Forward stress testing and the NPV-aware scorecard:__ The hybrid Monte Carlo turns a single expected-return number into a full distribution of outcomes, and the tail-focused scorecard (VaR, CVaR, max drawdown) plus portfolio NPV reveals where classical mean-variance breaks down. The portfolio NPV relative to the risk-free rate is the economically correct measure of failure — a portfolio that grew in nominal terms but underperformed risk-free has still failed.

__What comes next?__ In [Session 2](../session-2/eCornell-AI-Finance-S2-Lecture-AIRebalancingEngine-May-2026.ipynb), we move from static weights to adaptive allocation, designing an AI rebalancing engine that uses Cobb-Douglas utility maximization with sentiment-driven preference weights (built on SIM parameters) to respond dynamically to changing market conditions. The Session 1 scorecard — computed in the stress-test example notebook and saved to `stress-test-baseline.jld2` — is the bar Session 2 must clear.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models. Real-world portfolio construction involves additional considerations including regulatory requirements, tax implications, liquidity constraints, and operational risks that are beyond the scope of this session.

___